<a href="https://colab.research.google.com/github/AlonsoRafael/analise-dados-pix/blob/main/pipeline_pre_processamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1º ETAPA



### Coleta de Dados
* Baixar datasets em formato CSV/JSON.
* Verificar se há valores faltantes e se a base é representativa.
* Realizar a limpeza.

In [17]:
import pandas as pd
import numpy as np

# 1. Carrega os dados
df = pd.read_csv('comprovantes_pix_10000_anomalias.csv', sep=';')

# 2. Converte a data e já limpa os dados irregulares (datas que não existem)
df['DataHora'] = pd.to_datetime(df['DataHora'], errors='coerce')
df = df.dropna(subset=['DataHora']).copy()

# 3. Cria as novas variáveis temporais
df['Hora_Transacao'] = df['DataHora'].dt.hour
df['Dia_Semana'] = df['DataHora'].dt.dayofweek
df['Fim_de_Semana'] = df['Dia_Semana'].apply(lambda x: 1 if x >= 5 else 0)
df['Horario_Comercial'] = df['Hora_Transacao'].apply(lambda x: 1 if 8 <= x <= 18 else 0)

# Dropa a coluna de data original
df_proc = df.drop(columns=['DataHora'])

print("Base limpa e variáveis temporais criadas. Tamanho:", df_proc.shape)

Base limpa e variáveis temporais criadas. Tamanho: (9976, 18)


### Pré-processamento
* Remoção de outliers (se necessário).

In [18]:
# Filtro IQR na coluna 'Valor'
q1 = df_proc['Valor'].quantile(0.25)
q3 = df_proc['Valor'].quantile(0.75)
iqr = q3 - q1

limite_inf = q1 - 1.5 * iqr
limite_sup = q3 + 1.5 * iqr

df_sem_outliers = df_proc[(df_proc['Valor'] >= limite_inf) & (df_proc['Valor'] <= limite_sup)].copy()

print(f"Linhas mantidas após remoção de outliers: {len(df_sem_outliers)}")

Linhas mantidas após remoção de outliers: 9976


* Transformações
* Tratamento de atributos

In [22]:
from sklearn.preprocessing import LabelEncoder

# Separa só as colunas que têm texto (strings)
cols_categoricas = df_sem_outliers.select_dtypes(include=['object']).columns
le = LabelEncoder()

df_enc = df_sem_outliers.copy()

# Passa em cada coluna de texto convertendo as categorias em números (ex: Nubank = 1, Itaú = 2)
for col in cols_categoricas:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

print("Base após converter textos para números (Label Encoder):")
display(df_enc.head(3))

Base após converter textos para números (Label Encoder):


,EndToEndId,Valor,Moeda,Pagador_Nome,Pagador_CPF_CNPJ,Pagador_Banco,Recebedor_Nome,Recebedor_CPF_CNPJ,Recebedor_Banco,ChavePix_Utilizada,TipoChave,Descricao,Status,Anomalia,Hora_Transacao,Dia_Semana,Fim_de_Semana,Horario_Comercial
0,9370,4658.86,0,5579,362,0,230,9001,11,1861,4,920,0,0,21,3,0,0
1,9363,3184.72,0,731,7485,5,1152,2061,10,280,4,946,2,0,17,0,0,1
2,7570,1054.48,0,4317,3851,1,4738,4828,7,7473,2,83,2,0,15,4,0,1


* Normalização e padronização dos dados: StandardScaler vs MinMaxScaler

Aplicação de StandardScaler e MinMaxScaler nas variáveis numéricas (incluindo as novas *features* de tempo) para testar qual se adequa melhor ao modelo.

In [23]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Pega as colunas numéricas, mas tira a 'Anomalia' pra não escalar o que queremos prever
cols_num = df_enc.select_dtypes(include=[np.number]).columns
cols_num = cols_num.drop('Anomalia', errors='ignore')

df_std = df_enc.copy()
df_mm = df_enc.copy()

# 1. StandardScaler: deixa a média em 0 e desvio padrão em 1
scaler_std = StandardScaler()
df_std[cols_num] = scaler_std.fit_transform(df_enc[cols_num])

# 2. MinMaxScaler: espreme todos os valores pra ficarem no intervalo entre 0 e 1
scaler_mm = MinMaxScaler()
df_mm[cols_num] = scaler_mm.fit_transform(df_enc[cols_num])

# Separando algumas colunas só pra printar e comparar o efeito das escalas
cols_exemplo = ['Valor', 'Hora_Transacao', 'Dia_Semana']

print("Resultado do StandardScaler")
display(df_std[cols_exemplo].head(3))

print("\nResultado do MinMaxScaler")
display(df_mm[cols_exemplo].head(3))

Resultado do StandardScaler


,Valor,Hora_Transacao,Dia_Semana
0,1.479344,1.360262,0.003111
1,0.466637,0.784525,-1.474677
2,-0.996800,0.496657,0.495707



Resultado do MinMaxScaler


,Valor,Hora_Transacao,Dia_Semana
0,0.931811,0.913043,0.500000
1,0.636971,0.739130,0.000000
2,0.210905,0.652174,0.666667
